In [2]:
%pip install pymongo mysql-connector-python pandas

import mysql.connector
from pymongo import MongoClient
import json
import os
import datetime
import pandas as pd

# Conexión MySQL
mysql_conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="saludusm"
)
mysql_cursor = mysql_conn.cursor(dictionary=True)

# Conexión MongoDB (Replica Set)
uri = "mongodb://192.168.1.8:30001,192.168.1.8:30002,192.168.1.8:30003/?replicaSet=rs0"
mongo_client = MongoClient(uri, serverSelectionTimeoutMS=5000)
db_mongo = mongo_client["salud_usm"]

try:
    print("Primario:", mongo_client.primary)
    print("Nodos:", list(mongo_client.nodes))
    print("Conexión a MongoDB establecida")
except Exception as e:
    print(f"Error: {e}")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Primario: None
Nodos: []
Conexión a MongoDB establecida


In [45]:
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("INICIALIZANDO DATOS PARA TODAS LAS CONSULTAS")


colecciones = ['prevision', 'especialidad', 'estado_cita', 'centro_medico', 
               'paciente', 'medico', 'cita', 'usuario']
for col in colecciones:
    db_mongo[col].drop()
print("MongoDB limpiado.")

mysql_cursor.execute("SET FOREIGN_KEY_CHECKS = 0")
tablas = ['diagnostico_atencion', 'receta_linea', 'atencion', 'cita', 
          'medico_especialidad', 'medico_centro', 'medico', 'paciente', 
          'usuario', 'centro_medico', 'diagnostico', 'medicamento', 
          'especialidad', 'estado_cita', 'prevision']
for tabla in tablas:
    mysql_cursor.execute(f"TRUNCATE TABLE {tabla}")
mysql_cursor.execute("SET FOREIGN_KEY_CHECKS = 1")
mysql_conn.commit()
print("MySQL limpiado.")

print("\nInsertando catálogos...")

previsiones = [
    {"id_prevision": 1, "nombre": "Fonasa"},
    {"id_prevision": 2, "nombre": "Isapre"},
    {"id_prevision": 3, "nombre": "Particular"}
]
db_mongo.prevision.insert_many(previsiones)
mysql_cursor.executemany("INSERT INTO prevision (id_prevision, nombre) VALUES (%s, %s)", 
                         [(p["id_prevision"], p["nombre"]) for p in previsiones])

especialidades = [
    {"id_especialidad": 1, "nombre": "Medicina General"},
    {"id_especialidad": 2, "nombre": "Cardiología"},
    {"id_especialidad": 3, "nombre": "Pediatría"},
    {"id_especialidad": 4, "nombre": "Dermatología"},
    {"id_especialidad": 5, "nombre": "Traumatología"},
    {"id_especialidad": 6, "nombre": "Neurología"},
    {"id_especialidad": 7, "nombre": "Oftalmología"},
    {"id_especialidad": 8, "nombre": "Ginecología"}
]
db_mongo.especialidad.insert_many(especialidades)
mysql_cursor.executemany("INSERT INTO especialidad (id_especialidad, nombre) VALUES (%s, %s)", 
                         [(e["id_especialidad"], e["nombre"]) for e in especialidades])

estados = [
    {"id_estado": 1, "nombre": "Reservada"},
    {"id_estado": 2, "nombre": "Confirmada"},
    {"id_estado": 3, "nombre": "Atendida"},
    {"id_estado": 4, "nombre": "No Asistio"},
    {"id_estado": 5, "nombre": "Cancelada"}
]
db_mongo.estado_cita.insert_many(estados)
mysql_cursor.executemany("INSERT INTO estado_cita (id_estado, nombre) VALUES (%s, %s)", 
                         [(e["id_estado"], e["nombre"]) for e in estados])

centros = [
    {"id_centro": 1, "codigo": "C001", "nombre": "Centro SaludUSM Casa Central", "comuna": "Valparaíso", "region": "Valparaíso"},
    {"id_centro": 2, "codigo": "C002", "nombre": "Centro SaludUSM San Joaquín", "comuna": "San Joaquín", "region": "Metropolitana"},
    {"id_centro": 3, "codigo": "C003", "nombre": "Centro SaludUSM Maipú", "comuna": "Maipú", "region": "Metropolitana"},
    {"id_centro": 4, "codigo": "C004", "nombre": "Centro SaludUSM Concepción", "comuna": "Concepción", "region": "Biobío"}
]
db_mongo.centro_medico.insert_many(centros)
mysql_cursor.executemany("INSERT INTO centro_medico (id_centro, codigo, nombre, comuna, region) VALUES (%s, %s, %s, %s, %s)", 
                         [(c["id_centro"], c["codigo"], c["nombre"], c["comuna"], c["region"]) for c in centros])

mysql_conn.commit()
print("Catálogos insertados.")


print("\nInsertando pacientes...")

pacientes = [
    {"id_paciente": 1, "rut": "11111111-1", "nombre": "Juan", "apellido": "Pérez", "email": "juan@correo.cl", "telefono": "911111111", "sexo": "M", "id_prevision": 2},
    {"id_paciente": 2, "rut": "22222222-2", "nombre": "María", "apellido": "López", "email": "maria@correo.cl", "telefono": "922222222", "sexo": "F", "id_prevision": 1},
    {"id_paciente": 3, "rut": "33333333-3", "nombre": "Carlos", "apellido": "Silva", "email": "carlos@correo.cl", "telefono": "933333333", "sexo": "M", "id_prevision": 3}
]

for p in pacientes:
    db_mongo.paciente.insert_one(p)
    mysql_cursor.execute("""
        INSERT INTO usuario (email, rut, nombre, apellido, telefono, fecha_nacimiento, sexo, password_hash, rol)
        VALUES (%s, %s, %s, %s, %s, '1990-01-01', %s, 'hash', 'PACIENTE')
    """, (p["email"], p["rut"], p["nombre"], p["apellido"], p["telefono"], p["sexo"]))
    mysql_cursor.execute("INSERT INTO paciente (id_paciente, email, id_prevision) VALUES (%s, %s, %s)", 
                         (p["id_paciente"], p["email"], p["id_prevision"]))


print("Insertando médicos...")

medicos = [
    {"id_medico": 1, "rut": "15000001-7", "nombre": "Pedro", "apellido": "Araya", "email": "pedro.araya@saludusm.cl", "telefono": "915000017", "sexo": "M", "id_especialidades": [1, 2]},
    {"id_medico": 2, "rut": "16000002-8", "nombre": "Ana", "apellido": "Soto", "email": "ana.soto@saludusm.cl", "telefono": "916000028", "sexo": "F", "id_especialidades": [3]}
]

for m in medicos:
    db_mongo.medico.insert_one(m)
    mysql_cursor.execute("""
        INSERT INTO usuario (email, rut, nombre, apellido, telefono, fecha_nacimiento, sexo, password_hash, rol)
        VALUES (%s, %s, %s, %s, %s, '1980-01-01', %s, 'hash', 'MEDICO')
    """, (m["email"], m["rut"], m["nombre"], m["apellido"], m["telefono"], m["sexo"]))
    mysql_cursor.execute("INSERT INTO medico (id_medico, email) VALUES (%s, %s)", (m["id_medico"], m["email"]))
    for id_esp in m["id_especialidades"]:
        mysql_cursor.execute("INSERT INTO medico_especialidad (id_medico, id_especialidad) VALUES (%s, %s)", (m["id_medico"], id_esp))

mysql_conn.commit()
print("Pacientes y médicos insertados.")


print("\nInsertando citas y atenciones...")


citas = [
   
    {
        "id_cita": 1, "id_paciente": 1, "id_medico": 1, "id_especialidad": 2, "id_centro": 1,
        "fecha_hora": datetime(2026, 11, 15, 10, 0), "id_estado": 3,
        "atencion": {
            "motivo": "Dolor en el pecho",
            "observaciones": "Paciente con hipertensión",
            "diagnosticos": [{"codigo_cie10": "I10", "descripcion": "Hipertensión esencial"}],
            "recetas": [{"medicamento": "Losartán", "dosis": "50 mg cada 24 horas", "dias_tratamiento": 30}]
        }
    },
    {
        "id_cita": 2, "id_paciente": 2, "id_medico": 1, "id_especialidad": 1, "id_centro": 2,
        "fecha_hora": datetime(2026, 11, 15, 11, 0), "id_estado": 3,
        "atencion": {
            "motivo": "Control de presión",
            "observaciones": "Presión 140/90",
            "diagnosticos": [{"codigo_cie10": "I10", "descripcion": "Hipertensión esencial"}],
            "recetas": [{"medicamento": "Enalapril", "dosis": "10 mg cada 12 horas", "dias_tratamiento": 30}]
        }
    },

    {
        "id_cita": 3, "id_paciente": 1, "id_medico": 1, "id_especialidad": 1, "id_centro": 1,
        "fecha_hora": datetime(2026, 11, 16, 10, 0), "id_estado": 3,
        "atencion": {
            "motivo": "Control general",
            "observaciones": "Paciente estable",
            "diagnosticos": [{"codigo_cie10": "Z00.0", "descripcion": "Examen médico general"}],
            "recetas": [{"medicamento": "Paracetamol", "dosis": "500 mg", "dias_tratamiento": 5}]
        }
    },
    {
        "id_cita": 4, "id_paciente": 1, "id_medico": 2, "id_especialidad": 3, "id_centro": 3,
        "fecha_hora": datetime(2026, 11, 17, 10, 0), "id_estado": 3,
        "atencion": {
            "motivo": "Control infantil",
            "observaciones": "Paciente pediátrico",
            "diagnosticos": [{"codigo_cie10": "Z00.1", "descripcion": "Examen de salud infantil"}],
            "recetas": [{"medicamento": "Amoxicilina", "dosis": "500 mg", "dias_tratamiento": 7}]
        }
    },

    {
        "id_cita": 5, "id_paciente": 2, "id_medico": 1, "id_especialidad": 1, "id_centro": 1,
        "fecha_hora": datetime(2026, 11, 18, 10, 0), "id_estado": 4  # No Asistio
    },
    {
        "id_cita": 6, "id_paciente": 3, "id_medico": 1, "id_especialidad": 1, "id_centro": 1,
        "fecha_hora": datetime(2026, 11, 19, 10, 0), "id_estado": 4  # No Asistio
    },
    {
        "id_cita": 7, "id_paciente": 3, "id_medico": 1, "id_especialidad": 1, "id_centro": 1,
        "fecha_hora": datetime(2026, 11, 20, 10, 0), "id_estado": 1  # Reservada
    }
]

for c in citas:
    db_mongo.cita.insert_one(c)
    mysql_cursor.execute("""
        INSERT INTO cita (id_cita, id_paciente, id_medico, id_especialidad, id_centro, fecha_hora, id_estado)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (c["id_cita"], c["id_paciente"], c["id_medico"], c["id_especialidad"], 
          c["id_centro"], c["fecha_hora"], c["id_estado"]))
    
    if "atencion" in c:
        mysql_cursor.execute("""
            INSERT INTO atencion (id_cita, motivo, observaciones)
            VALUES (%s, %s, %s)
        """, (c["id_cita"], c["atencion"]["motivo"], c["atencion"]["observaciones"]))
        id_atencion = mysql_cursor.lastrowid
        
        for d in c["atencion"]["diagnosticos"]:
            mysql_cursor.execute("INSERT IGNORE INTO diagnostico (codigo_cie10, descripcion) VALUES (%s, %s)", 
                                 (d["codigo_cie10"], d["descripcion"]))
            mysql_cursor.execute("SELECT id_diagnostico FROM diagnostico WHERE codigo_cie10 = %s", (d["codigo_cie10"],))
            id_diag = mysql_cursor.fetchone()["id_diagnostico"]
            mysql_cursor.execute("INSERT IGNORE INTO diagnostico_atencion (id_atencion, id_diagnostico) VALUES (%s, %s)", 
                                 (id_atencion, id_diag))
            
        for r in c["atencion"]["recetas"]:
            mysql_cursor.execute("INSERT IGNORE INTO medicamento (nombre) VALUES (%s)", (r["medicamento"],))
            mysql_cursor.execute("SELECT id_medicamento FROM medicamento WHERE nombre = %s", (r["medicamento"],))
            id_med = mysql_cursor.fetchone()["id_medicamento"]
            mysql_cursor.execute("""
                INSERT INTO receta_linea (id_atencion, id_medicamento, dosis, dias_tratamiento)
                VALUES (%s, %s, %s, %s)
            """, (id_atencion, id_med, r["dosis"], r["dias_tratamiento"]))

mysql_conn.commit()
print("Citas y atenciones insertadas.")



print("VERIFICACIÓN DE DATOS")

print(f"MongoDB - Citas: {db_mongo.cita.count_documents({})}")
mysql_cursor.execute("SELECT COUNT(*) as total FROM cita")
print(f"MySQL - Citas: {mysql_cursor.fetchone()['total']}")

print("\nDatos inicializados para todas las consultas.")

INICIALIZANDO DATOS PARA TODAS LAS CONSULTAS
MongoDB limpiado.
MySQL limpiado.

Insertando catálogos...
Catálogos insertados.

Insertando pacientes...
Insertando médicos...
Pacientes y médicos insertados.

Insertando citas y atenciones...
Citas y atenciones insertadas.
VERIFICACIÓN DE DATOS
MongoDB - Citas: 7
MySQL - Citas: 7

Datos inicializados para todas las consultas.


In [43]:
from datetime import datetime

print("CONSULTA (a): Agenda diaria de un médico")


rut_medico = "15000001-7"
fecha = "2026-11-15"

medico = db_mongo.medico.find_one({"rut": rut_medico})
id_medico = medico["id_medico"]
print(f"Médico: {medico['nombre']} {medico['apellido']} (id: {id_medico})")

fecha_inicio = datetime.strptime(fecha, "%Y-%m-%d")
fecha_fin = fecha_inicio.replace(hour=23, minute=59, second=59)

pipeline_a = [
    {"$match": {"id_medico": id_medico, "fecha_hora": {"$gte": fecha_inicio, "$lte": fecha_fin}}},
    {"$lookup": {"from": "paciente", "localField": "id_paciente", "foreignField": "id_paciente", "as": "pac_info"}},
    {"$unwind": "$pac_info"},
    {"$lookup": {"from": "prevision", "localField": "pac_info.id_prevision", "foreignField": "id_prevision", "as": "prev_info"}},
    {"$unwind": "$prev_info"},
    {"$lookup": {"from": "centro_medico", "localField": "id_centro", "foreignField": "id_centro", "as": "centro_info"}},
    {"$unwind": "$centro_info"},
    {"$lookup": {"from": "especialidad", "localField": "id_especialidad", "foreignField": "id_especialidad", "as": "esp_info"}},
    {"$unwind": "$esp_info"},
    {"$sort": {"fecha_hora": 1}},
    {"$project": {
        "_id": 0,
        "hora": {"$dateToString": {"format": "%H:%M", "date": "$fecha_hora"}},
        "paciente": {"$concat": ["$pac_info.nombre", " ", "$pac_info.apellido"]},
        "prevision": "$prev_info.nombre",
        "centro": "$centro_info.nombre",
        "especialidad": "$esp_info.nombre",
        "estado": "$id_estado"
    }}
]
res_a_mongo = list(db_mongo.cita.aggregate(pipeline_a))
print(pd.DataFrame(res_a_mongo))

CONSULTA (a): Agenda diaria de un médico
Médico: Pedro Araya (id: 1)
    hora     paciente prevision                        centro  \
0  10:00   Juan Pérez    Isapre  Centro SaludUSM Casa Central   
1  11:00  María López    Fonasa   Centro SaludUSM San Joaquín   

       especialidad  estado  
0       Cardiología       3  
1  Medicina General       3  


In [42]:
from datetime import datetime

print("CONSULTA (a): Agenda diaria de un médico")


rut_medico = "15000001-7"
fecha = "2026-11-15"

medico = db_mongo.medico.find_one({"rut": rut_medico})
if not medico:
    print(f"No se encontró el médico con RUT {rut_medico}")
else:
    id_medico = medico["id_medico"]
    print(f"Médico: {medico['nombre']} {medico['apellido']} (id: {id_medico})")

    fecha_inicio = datetime.strptime(fecha, "%Y-%m-%d")
    fecha_fin = fecha_inicio.replace(hour=23, minute=59, second=59)

    pipeline_a = [
        {"$match": {"id_medico": id_medico, "fecha_hora": {"$gte": fecha_inicio, "$lte": fecha_fin}}},
        {"$lookup": {"from": "paciente", "localField": "id_paciente", "foreignField": "id_paciente", "as": "pac_info"}},
        {"$unwind": "$pac_info"},
        {"$lookup": {"from": "prevision", "localField": "pac_info.id_prevision", "foreignField": "id_prevision", "as": "prev_info"}},
        {"$unwind": "$prev_info"},
        {"$lookup": {"from": "centro_medico", "localField": "id_centro", "foreignField": "id_centro", "as": "centro_info"}},
        {"$unwind": "$centro_info"},
        {"$lookup": {"from": "especialidad", "localField": "id_especialidad", "foreignField": "id_especialidad", "as": "esp_info"}},
        {"$unwind": "$esp_info"},
        {"$sort": {"fecha_hora": 1}},
        {"$project": {
            "_id": 0,
            "hora": {"$dateToString": {"format": "%H:%M", "date": "$fecha_hora"}},
            "paciente": {"$concat": ["$pac_info.nombre", " ", "$pac_info.apellido"]},
            "prevision": "$prev_info.nombre",
            "centro": "$centro_info.nombre",
            "especialidad": "$esp_info.nombre",
            "estado": "$id_estado"
        }}
    ]

    res_a_mongo = list(db_mongo.cita.aggregate(pipeline_a))
    print(pd.DataFrame(res_a_mongo))

CONSULTA (a): Agenda diaria de un médico
Médico: Pedro Araya (id: 1)
    hora     paciente prevision                        centro  \
0  10:00   Juan Pérez    Isapre  Centro SaludUSM Casa Central   
1  11:00  María López    Fonasa   Centro SaludUSM San Joaquín   

       especialidad  estado  
0       Cardiología       3  
1  Medicina General       3  


In [41]:
print("\nSQL Equivalente ")
sql_a = """
SELECT 
    DATE_FORMAT(c.fecha_hora, '%H:%i') AS hora,
    CONCAT(p.nombre, ' ', p.apellido) AS paciente,
    pv.nombre AS prevision,
    cm.nombre AS centro,
    e.nombre AS especialidad,
    ec.nombre AS estado
FROM cita c
JOIN paciente pa ON c.id_paciente = pa.id_paciente
JOIN usuario p ON pa.email = p.email
JOIN prevision pv ON pa.id_prevision = pv.id_prevision
JOIN centro_medico cm ON c.id_centro = cm.id_centro
JOIN especialidad e ON c.id_especialidad = e.id_especialidad
JOIN estado_cita ec ON c.id_estado = ec.id_estado
JOIN medico m ON c.id_medico = m.id_medico
JOIN usuario um ON m.email = um.email
WHERE um.rut = %s AND DATE(c.fecha_hora) = %s
ORDER BY c.fecha_hora;
"""
df_a_sql = pd.read_sql(sql_a, con=mysql_conn, params=(rut_medico, fecha))
print(df_a_sql)


SQL Equivalente 
    hora     paciente prevision                        centro  \
0  10:00   Juan Pérez    Isapre  Centro SaludUSM Casa Central   
1  11:00  María López    Fonasa   Centro SaludUSM San Joaquín   

       especialidad    estado  
0       Cardiología  Atendida  
1  Medicina General  Atendida  


In [46]:

print("CONSULTA (b): Historial clínico de un paciente")


rut_paciente = "11111111-1"
paciente = db_mongo.paciente.find_one({"rut": rut_paciente})
id_paciente = paciente["id_paciente"]
print(f"Paciente: {paciente['nombre']} {paciente['apellido']} (id: {id_paciente})")

pipeline_b = [
    {"$match": {"id_paciente": id_paciente}},
    {"$lookup": {"from": "medico", "localField": "id_medico", "foreignField": "id_medico", "as": "med_info"}},
    {"$unwind": "$med_info"},
    {"$lookup": {"from": "centro_medico", "localField": "id_centro", "foreignField": "id_centro", "as": "centro_info"}},
    {"$unwind": "$centro_info"},
    {"$lookup": {"from": "especialidad", "localField": "id_especialidad", "foreignField": "id_especialidad", "as": "esp_info"}},
    {"$unwind": "$esp_info"},
    {"$sort": {"fecha_hora": -1}},
    {"$project": {
        "_id": 0,
        "fecha": "$fecha_hora",
        "centro": "$centro_info.nombre",
        "especialidad": "$esp_info.nombre",
        "medico": {"$concat": ["$med_info.nombre", " ", "$med_info.apellido"]},
        "motivo": {"$ifNull": ["$atencion.motivo", ""]},
        "diagnosticos": {"$ifNull": ["$atencion.diagnosticos", []]},
        "recetas": {"$ifNull": ["$atencion.recetas", []]}
    }}
]
res_b_mongo = list(db_mongo.cita.aggregate(pipeline_b))
print(pd.DataFrame(res_b_mongo))

CONSULTA (b): Historial clínico de un paciente
Paciente: Juan Pérez (id: 1)
                fecha                        centro      especialidad  \
0 2026-11-17 10:00:00         Centro SaludUSM Maipú         Pediatría   
1 2026-11-16 10:00:00  Centro SaludUSM Casa Central  Medicina General   
2 2026-11-15 10:00:00  Centro SaludUSM Casa Central       Cardiología   

        medico             motivo  \
0     Ana Soto   Control infantil   
1  Pedro Araya    Control general   
2  Pedro Araya  Dolor en el pecho   

                                        diagnosticos  \
0  [{'codigo_cie10': 'Z00.1', 'descripcion': 'Exa...   
1  [{'codigo_cie10': 'Z00.0', 'descripcion': 'Exa...   
2  [{'codigo_cie10': 'I10', 'descripcion': 'Hiper...   

                                             recetas  
0  [{'medicamento': 'Amoxicilina', 'dosis': '500 ...  
1  [{'medicamento': 'Paracetamol', 'dosis': '500 ...  
2  [{'medicamento': 'Losartán', 'dosis': '50 mg c...  


In [47]:
print("\nSQL Equivalente")
sql_b = """
SELECT 
    c.fecha_hora AS fecha,
    cm.nombre AS centro,
    e.nombre AS especialidad,
    CONCAT(um.nombre, ' ', um.apellido) AS medico,
    a.motivo,
    GROUP_CONCAT(CONCAT(d.codigo_cie10, ' - ', d.descripcion) SEPARATOR '; ') AS diagnosticos,
    GROUP_CONCAT(CONCAT(m.nombre, ' - ', r.dosis, ' - ', r.dias_tratamiento, 'd') SEPARATOR '; ') AS recetas
FROM cita c
JOIN paciente pa ON c.id_paciente = pa.id_paciente
JOIN usuario up ON pa.email = up.email
JOIN medico me ON c.id_medico = me.id_medico
JOIN usuario um ON me.email = um.email
JOIN centro_medico cm ON c.id_centro = cm.id_centro
JOIN especialidad e ON c.id_especialidad = e.id_especialidad
LEFT JOIN atencion a ON a.id_cita = c.id_cita
LEFT JOIN diagnostico_atencion da ON da.id_atencion = a.id_atencion
LEFT JOIN diagnostico d ON da.id_diagnostico = d.id_diagnostico
LEFT JOIN receta_linea r ON r.id_atencion = a.id_atencion
LEFT JOIN medicamento m ON r.id_medicamento = m.id_medicamento
WHERE up.rut = %s
GROUP BY c.id_cita, c.fecha_hora, cm.nombre, e.nombre, um.nombre, um.apellido, a.motivo
ORDER BY c.fecha_hora DESC;
"""
df_b_sql = pd.read_sql(sql_b, con=mysql_conn, params=(rut_paciente,))
print(df_b_sql)


SQL Equivalente
                fecha                        centro      especialidad  \
0 2026-11-17 10:00:00         Centro SaludUSM Maipú         Pediatría   
1 2026-11-16 10:00:00  Centro SaludUSM Casa Central  Medicina General   
2 2026-11-15 10:00:00  Centro SaludUSM Casa Central       Cardiología   

        medico             motivo                      diagnosticos  \
0     Ana Soto   Control infantil  Z00.1 - Examen de salud infantil   
1  Pedro Araya    Control general     Z00.0 - Examen médico general   
2  Pedro Araya  Dolor en el pecho       I10 - Hipertensión esencial   

                                recetas  
0             Amoxicilina - 500 mg - 7d  
1             Paracetamol - 500 mg - 5d  
2  Losartán - 50 mg cada 24 horas - 30d  


In [48]:

print("CONSULTA (c): Tasa de inasistencia por centro")

pipeline_c = [
    {"$lookup": {"from": "estado_cita", "localField": "id_estado", "foreignField": "id_estado", "as": "estado_info"}},
    {"$unwind": "$estado_info"},
    {"$group": {
        "_id": "$id_centro",
        "total_citas": {"$sum": 1},
        "no_asistio": {"$sum": {"$cond": [{"$eq": ["$estado_info.nombre", "No Asistio"]}, 1, 0]}}
    }},
    {"$project": {
        "total_citas": 1,
        "no_asistio": 1,
        "porcentaje": {"$round": [{"$multiply": [{"$divide": ["$no_asistio", "$total_citas"]}, 100]}, 2]}
    }},
    {"$match": {"porcentaje": {"$gt": 15}}},
    {"$sort": {"porcentaje": -1}},
    {"$lookup": {"from": "centro_medico", "localField": "_id", "foreignField": "id_centro", "as": "centro_info"}},
    {"$unwind": "$centro_info"},
    {"$project": {"_id": 0, "centro": "$centro_info.nombre", "total_citas": 1, "no_asistio": 1, "porcentaje": 1}}
]
res_c_mongo = list(db_mongo.cita.aggregate(pipeline_c))
print(pd.DataFrame(res_c_mongo))

CONSULTA (c): Tasa de inasistencia por centro
   total_citas  no_asistio  porcentaje                        centro
0            5           2        40.0  Centro SaludUSM Casa Central


In [49]:
print("\nSQL Equivalente")
sql_c = """
SELECT 
    cm.nombre AS centro,
    COUNT(c.id_cita) AS total_citas,
    SUM(CASE WHEN ec.nombre = 'No Asistio' THEN 1 ELSE 0 END) AS no_asistio,
    ROUND(
        (SUM(CASE WHEN ec.nombre = 'No Asistio' THEN 1 ELSE 0 END) / COUNT(c.id_cita)) * 100, 
        2
    ) AS porcentaje
FROM centro_medico cm
JOIN cita c ON cm.id_centro = c.id_centro
JOIN estado_cita ec ON c.id_estado = ec.id_estado
GROUP BY cm.id_centro, cm.nombre
HAVING porcentaje > 15
ORDER BY porcentaje DESC;
"""
df_c_sql = pd.read_sql(sql_c, con=mysql_conn)
print(df_c_sql)


SQL Equivalente
                         centro  total_citas  no_asistio  porcentaje
0  Centro SaludUSM Casa Central            5         2.0        40.0


In [50]:

print("CONSULTA (d): Ranking de medicamentos por especialidad")


especialidad_buscada = "Medicina General"

pipeline_d = [
    {"$lookup": {"from": "especialidad", "localField": "id_especialidad", "foreignField": "id_especialidad", "as": "esp_info"}},
    {"$unwind": "$esp_info"},
    {"$match": {"esp_info.nombre": especialidad_buscada}},
    {"$unwind": "$atencion.recetas"},
    {"$group": {"_id": "$atencion.recetas.medicamento", "total": {"$sum": 1}}},
    {"$sort": {"total": -1, "_id": 1}},
    {"$limit": 5},
    {"$project": {"_id": 0, "medicamento": "$_id", "total": 1}}
]
res_d_mongo = list(db_mongo.cita.aggregate(pipeline_d))
print(pd.DataFrame(res_d_mongo))

CONSULTA (d): Ranking de medicamentos por especialidad
   total  medicamento
0      1    Enalapril
1      1  Paracetamol


In [51]:
print("\nSQL Equivalente")
sql_d = """
SELECT 
    m.nombre AS medicamento,
    COUNT(*) AS total
FROM cita c
JOIN especialidad e ON c.id_especialidad = e.id_especialidad
JOIN atencion a ON a.id_cita = c.id_cita
JOIN receta_linea r ON r.id_atencion = a.id_atencion
JOIN medicamento m ON r.id_medicamento = m.id_medicamento
WHERE e.nombre = %s
GROUP BY m.id_medicamento, m.nombre
ORDER BY total DESC, m.nombre ASC
LIMIT 5;
"""
df_d_sql = pd.read_sql(sql_d, con=mysql_conn, params=(especialidad_buscada,))
print(df_d_sql)


SQL Equivalente
   medicamento  total
0    Enalapril      1
1  Paracetamol      1


In [52]:

print("CONSULTA (e): Pacientes multi-especialidad")


pipeline_e = [
    {"$lookup": {"from": "estado_cita", "localField": "id_estado", "foreignField": "id_estado", "as": "estado_info"}},
    {"$unwind": "$estado_info"},
    {"$match": {"estado_info.nombre": "Atendida"}},
    {"$lookup": {"from": "paciente", "localField": "id_paciente", "foreignField": "id_paciente", "as": "pac_info"}},
    {"$unwind": "$pac_info"},
    {"$lookup": {"from": "especialidad", "localField": "id_especialidad", "foreignField": "id_especialidad", "as": "esp_info"}},
    {"$unwind": "$esp_info"},
    {"$group": {
        "_id": "$id_paciente",
        "nombre": {"$first": {"$concat": ["$pac_info.nombre", " ", "$pac_info.apellido"]}},
        "especialidades": {"$addToSet": "$esp_info.nombre"}
    }},
    {"$match": {"$expr": {"$gte": [{"$size": "$especialidades"}, 3]}}},
    {"$project": {"_id": 0, "nombre": 1, "cantidad": {"$size": "$especialidades"}, "especialidades": 1}},
    {"$sort": {"cantidad": -1}}
]
res_e_mongo = list(db_mongo.cita.aggregate(pipeline_e))
print(pd.DataFrame(res_e_mongo))

CONSULTA (e): Pacientes multi-especialidad
       nombre                              especialidades  cantidad
0  Juan Pérez  [Medicina General, Pediatría, Cardiología]         3


In [53]:
print("\n--- SQL Equivalente ---")
sql_e = """
SELECT 
    CONCAT(u.nombre, ' ', u.apellido) AS paciente,
    COUNT(DISTINCT c.id_especialidad) AS cantidad,
    GROUP_CONCAT(DISTINCT e.nombre SEPARATOR ', ') AS especialidades
FROM cita c
JOIN estado_cita ec ON c.id_estado = ec.id_estado
JOIN paciente p ON c.id_paciente = p.id_paciente
JOIN usuario u ON p.email = u.email
JOIN especialidad e ON c.id_especialidad = e.id_especialidad
WHERE ec.nombre = 'Atendida'
GROUP BY p.id_paciente, u.nombre, u.apellido
HAVING cantidad >= 3
ORDER BY cantidad DESC;
"""
df_e_sql = pd.read_sql(sql_e, con=mysql_conn)
print(df_e_sql)


--- SQL Equivalente ---
     paciente  cantidad                            especialidades
0  Juan Pérez         3  Cardiología, Medicina General, Pediatría


In [54]:
print("\n--- SQL Equivalente ---")
sql_e = """
SELECT 
    CONCAT(u.nombre, ' ', u.apellido) AS paciente,
    COUNT(DISTINCT c.id_especialidad) AS cantidad,
    GROUP_CONCAT(DISTINCT e.nombre SEPARATOR ', ') AS especialidades
FROM cita c
JOIN estado_cita ec ON c.id_estado = ec.id_estado
JOIN paciente p ON c.id_paciente = p.id_paciente
JOIN usuario u ON p.email = u.email
JOIN especialidad e ON c.id_especialidad = e.id_especialidad
WHERE ec.nombre = 'Atendida'
GROUP BY p.id_paciente, u.nombre, u.apellido
HAVING cantidad >= 3
ORDER BY cantidad DESC;
"""
df_e_sql = pd.read_sql(sql_e, con=mysql_conn)
print(df_e_sql)


--- SQL Equivalente ---
     paciente  cantidad                            especialidades
0  Juan Pérez         3  Cardiología, Medicina General, Pediatría


In [55]:
import json
import os

carpeta_json = 'JSON'
os.makedirs(carpeta_json, exist_ok=True)

colecciones = db_mongo.list_collection_names()
print(f"Exportando {len(colecciones)} colecciones a '{carpeta_json}/'...")

for col in colecciones:
    datos = list(db_mongo[col].find({}, {'_id': 0}))
    ruta = os.path.join(carpeta_json, f"{col}.json")
    with open(ruta, 'w', encoding='utf-8') as f:
        json.dump(datos, f, ensure_ascii=False, indent=4, default=str)
    print(f" {col}.json ({len(datos)} documentos)")

print(f"\nArchivos JSON generados en la carpeta '{carpeta_json}'.")

Exportando 8 colecciones a 'JSON/'...
 centro_medico.json (4 documentos)
 cita.json (7 documentos)
 especialidad.json (8 documentos)
 prevision.json (3 documentos)
 test.json (2 documentos)
 paciente.json (3 documentos)
 medico.json (2 documentos)
 estado_cita.json (5 documentos)

Archivos JSON generados en la carpeta 'JSON'.
